In [13]:
import Gmsh: gmsh

In [14]:
# input parameters
const L = 6;                # Length
const B = 6;                # Breadth
const h = 0.1;              # Mesh size

x = (L/2);
y = (B/2);
r₁ = 2;
r₂ = 2.5;

a = x+r₁;
b = y+r₂;
c = x-r₁;
d = y-r₂;

In [15]:
gmsh.initialize()
gmsh.option.setNumber("General.Terminal", 1)

gmsh.option.setNumber("Mesh.Algorithm", 8)                  # Frontal-Delaunay
gmsh.option.setNumber("Mesh.RecombinationAlgorithm", 1)     # Blossom
gmsh.option.setNumber("Mesh.RecombineAll", 1)               # Recombine all surfaces (quadileteral mesh)

# Define points
gmsh.model.geo.addPoint(0, 0, 0, h, 1)
gmsh.model.geo.addPoint(L, 0, 0, h, 2)
gmsh.model.geo.addPoint(L, B, 0, h, 3)
gmsh.model.geo.addPoint(0, B, 0, h, 4)

# Add lines
gmsh.model.geo.addLine(1, 2, 1)
gmsh.model.geo.addLine(2, 3, 2)
gmsh.model.geo.addLine(3, 4, 3)
gmsh.model.geo.addLine(4, 1, 4)

# Create a rectangular curve loop and surface
Rectangle = gmsh.model.geo.addCurveLoop([1, 2, 3, 4])

# Construct Ellipse boundary points
gmsh.model.geo.addPoint(x, y,  0, h, 5)     # Center
gmsh.model.geo.addPoint(a, y,  0, h, 6)     # E1 (majorAxis)
gmsh.model.geo.addPoint(x, b,  0, h, 7)     # E2
gmsh.model.geo.addPoint(c, y,  0, h, 8)     # E3
gmsh.model.geo.addPoint(x, d,  0, h, 9)     # E4

# gmsh.model.geo.addEllipseArc(startTag, centerTag, majorAxisTag, endTag, tag=-1)
gmsh.model.geo.addEllipseArc(6, 5, 6, 7, 5)  
gmsh.model.geo.addEllipseArc(7, 5, 6, 8, 6)  
gmsh.model.geo.addEllipseArc(8, 5, 6, 9, 7)  
gmsh.model.geo.addEllipseArc(9, 5, 6, 6, 8)  

# Create circular curve loops
Ellipse = gmsh.model.geo.addCurveLoop([5, 6, 7, 8])

# Create plane surface
# 1 ⟹ Rectangle
# 2 ⟹ Ellipse
s1 = gmsh.model.geo.addPlaneSurface([1, 2])
s2 = gmsh.model.geo.addPlaneSurface([2])

# Domain
gmsh.model.addPhysicalGroup(2, [s1], 1)
gmsh.model.setPhysicalName(2, 1, "Ω₁")

# Domain
gmsh.model.addPhysicalGroup(2, [s2], 2)
gmsh.model.setPhysicalName(2, 2, "Ω₂")

# Boundary
gmsh.model.addPhysicalGroup(1, [5, 6, 7, 8], 3)
gmsh.model.setPhysicalName(1, 3, "Ω₀")

# Line
gmsh.model.addPhysicalGroup(1, [1], 4)
gmsh.model.setPhysicalName(1, 4, "A")

# Line
gmsh.model.addPhysicalGroup(1, [2], 5)
gmsh.model.setPhysicalName(1, 5, "B")

# Line
gmsh.model.addPhysicalGroup(1, [3], 6)
gmsh.model.setPhysicalName(1, 6, "C")

# Line
gmsh.model.addPhysicalGroup(1, [4], 7)
gmsh.model.setPhysicalName(1, 7, "D")

# Boundary
gmsh.model.addPhysicalGroup(1, [1, 2, 3, 4], 8)
gmsh.model.setPhysicalName(1, 8, "∂Ω")

# Synchronize geometry 
gmsh.model.geo.synchronize()

# Apply recombination explicitly (safe)
#gmsh.model.mesh.setRecombine(2, 1)_Quad
gmsh.model.mesh.generate(2)

# Create the folder (if it doesn't exist)
write_dir = joinpath(@__DIR__, "Model")
isdir(write_dir) || mkpath(write_dir)  

# Save mesh
gmsh.write(joinpath(write_dir, "2D_Plate_With_Elliptical_Interface.msh"))

# Launch GUI
gmsh.fltk.run() 

# Finalize Gmsh
gmsh.finalize()

Info    : Meshing 1D...
Info    : [  0%] Meshing curve 1 (Line)
Info    : [ 20%] Meshing curve 2 (Line)
Info    : [ 30%] Meshing curve 3 (Line)
Info    : [ 40%] Meshing curve 4 (Line)
Info    : [ 60%] Meshing curve 5 (Ellipse)
Info    : [ 70%] Meshing curve 6 (Ellipse)
Info    : [ 80%] Meshing curve 7 (Ellipse)
Info    : [ 90%] Meshing curve 8 (Ellipse)
Info    : Done meshing 1D (Wall 0.019093s, CPU 0.015625s)
Info    : Meshing 2D...
Info    : [  0%] Meshing surface 1 (Plane, Frontal-Delaunay for Quads)
Info    : [  0%] Blossom: 6114 internal 384 closed
Info    : [  0%] Blossom recombination completed (Wall 0.0932078s, CPU 0.09375s): 2095 quads, 0 triangles, 0 invalid quads, 0 quads with Q < 0.1, avg Q = 0.905514, min Q = 0.496747
Info    : [ 60%] Meshing surface 2 (Plane, Frontal-Delaunay for Quads)
Info    : [ 60%] Blossom: 4989 internal 144 closed
Info    : [ 60%] Blossom recombination completed (Wall 0.097003s, CPU 0.09375s): 1675 quads, 0 triangles, 0 invalid quads, 2 quads with Q

In [16]:
using GridapGmsh
using Gridap

In [17]:
# Check For Meshing
mesh_name = "2D_Plate_With_Elliptical_Interface.msh"
mesh_file = joinpath(@__DIR__, "Model", mesh_name)

isfile(mesh_file) || error("Path does not exist: $mesh_file")
model_check = GmshDiscreteModel(mesh_file)

result_path = joinpath(@__DIR__, "..", "..", "..", "Result", "Model_Check", "Quadileteral_Mesh", "2D_Plate_With_Hole")
isdir(result_path) || mkpath(result_path)

writevtk(model_check, joinpath(result_path, "model_check"))

Info    : Reading 'c:\Users\amiya\OneDrive\Desktop\Project\Model_Creation\Quadileteral_Gmsh_Model\Model\2D_Plate_With_Elliptical_Interface.msh'...
Info    : 19 entities
Info    : 3891 nodes
Info    : 4154 elements
Info    : Done reading 'c:\Users\amiya\OneDrive\Desktop\Project\Model_Creation\Quadileteral_Gmsh_Model\Model\2D_Plate_With_Elliptical_Interface.msh'


3-element Vector{Vector{String}}:
 ["c:\\Users\\amiya\\OneDrive\\Desktop\\Project\\Model_Creation\\Quadileteral_Gmsh_Model\\..\\..\\..\\Result\\Model_Check\\Quadileteral_Mesh\\2D_Plate_With_Hole\\model_check_0.vtu"]
 ["c:\\Users\\amiya\\OneDrive\\Desktop\\Project\\Model_Creation\\Quadileteral_Gmsh_Model\\..\\..\\..\\Result\\Model_Check\\Quadileteral_Mesh\\2D_Plate_With_Hole\\model_check_1.vtu"]
 ["c:\\Users\\amiya\\OneDrive\\Desktop\\Project\\Model_Creation\\Quadileteral_Gmsh_Model\\..\\..\\..\\Result\\Model_Check\\Quadileteral_Mesh\\2D_Plate_With_Hole\\model_check_2.vtu"]